# 📊 CICIoT2023 Deep Exploratory Data Analysis & Feature Engineering
## Phân Tích Thống Kê Chuyên Sâu, Trực Quan Hóa Đa Chiều & Khai Phá Đặc Trưng Lưu Lượng Mạng IoT
> **Đề tài Luận văn:** Nâng cao hiệu quả phát hiện tấn công mạng trong môi trường IoT dựa trên bộ dữ liệu CICIoT2023  
> **Mục tiêu Notebook:** Thấu hiểu bản chất vật lý và thống kê của 46+ đặc trưng mạng, phân tích 34 loại hình tấn công và 8 nhóm phân loại chuẩn IEEE Access, khảo sát tương quan, bất đối xứng lưu lượng, cờ TCP và đề xuất kỹ thuật trích xuất đặc trưng (Feature Engineering) phục vụ mô hình học máy.

---

### 📑 Mục Lục Chi Tiết:
1. [Thiết lập Môi trường, Thư viện & Cấu hình Đồ họa](#sec-01)
2. [Bộ nạp Dữ liệu Đa nền tảng & Tối ưu Bộ nhớ (CSV, Parquet, Pickle)](#sec-02)
3. [Kiểm toán Chất lượng Dữ liệu (Missing, Infinite, Zero-Inflated Values)](#sec-03)
4. [Phân tích Phân phối Nhãn Tấn công & Độ Mất Cân Bằng Nghiêm Trọng (34 → 8 Classes)](#sec-04)
5. [Phân tích Tốc độ Dòng & Hành vi Tấn công Lưu lượng Khối lượng lớn (Rate, Srate, Drate)](#sec-05)
6. [Giải phẫu Chữ ký Cờ TCP & Giao thức Mạng (TCP Flags Heatmap & Profile)](#sec-06)
7. [Phân phối Kích thước Gói tin & Độ Phân tán Thống kê (Min, Max, Mean, Std, Variance)](#sec-07)
8. [Phân tích Thời gian Đến giữa các Gói tin (Inter-Arrival Time - IAT & Botnet Jitter)](#sec-08)
9. [Đánh giá Sức mạnh Phân tách (Effect Size / Cohen's d: Benign vs Attacks)](#sec-09)
10. [Phân tích Đa cộng tuyến, Ma trận Tương quan & Phân cụm Phân cấp (Clustermap & Dendrogram)](#sec-10)
11. [Xếp hạng Tầm quan trọng Đặc trưng Đồng thuận (Consensus Feature Importance)](#sec-11)
12. [Trực quan hóa Không gian Chiều cao (PCA Scree & t-SNE 2D Manifold Projection)](#sec-12)
13. [Phân tích Dị biệt & Giá trị Ngoại lai (Outlier Behavior in Cybersecurity)](#sec-13)
14. [Khai phá Kỹ thuật Trích xuất Đặc trưng Mới (Domain-Specific Feature Engineering)](#sec-14)
15. [Tổng kết Tri thức Miền & Bảng Chữ ký Đặc trưng Tấn công](#sec-15)


<a id="sec-01"></a>
## 1. Thiết lập Môi trường, Thư viện & Cấu hình Đồ họa
* Khởi tạo các thư viện phân tích dữ liệu chuyên sâu: `numpy`, `pandas`, `scipy.stats`, `sklearn`, `joblib`.
* Cấu hình đồ thị Matplotlib/Seaborn với phong cách trực quan hóa báo cáo khoa học (High DPI, bảng màu hài hòa, lưới phụ, cỡ chữ chuẩn ấn bản).


In [ ]:
# ==============================================================================
# 1. SETUP & CONFIGURATION
# ==============================================================================
import os
import gc
import sys
import time
import warnings
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
from scipy.cluster.hierarchy import dendrogram, linkage

# Sklearn tools for EDA
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Filter warnings
warnings.filterwarnings('ignore')

# Global Constants
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Display helper
display_fn = globals().get('display', print)

# Plotting Aesthetics
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['figure.dpi'] = 120
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

# Color Palettes
COLOR_CYCLE = sns.color_palette("Set2", 10)
ATTACK_PALETTE = {
    'DDoS': '#e74c3c',
    'DoS': '#e67e22',
    'Recon': '#9b59b6',
    'Web-Based': '#3498db',
    'Mirai': '#d35400',
    'Spoofing': '#1abc9c',
    'BruteForce': '#f1c40f',
    'Normal': '#2ecc71',
    'Benign': '#2ecc71'
}

print("✅ Setup complete! Libraries imported and graphical themes configured.")
print(f"📦 Environment: Python {sys.version.split()[0]} | Pandas {pd.__version__} | NumPy {np.__version__}")


<a id="sec-02"></a>
## 2. Bộ nạp Dữ liệu Đa nền tảng & Tối ưu Bộ nhớ
* Hỗ trợ tự động nhận dạng môi trường làm việc: **Kaggle Notebooks**, **Google Colab**, hoặc **Local Machine**.
* Tự động quét và nạp dữ liệu ở các định dạng: `.csv` (chunked loading), `.parquet` (tối ưu tốc độ nạp), hoặc `.pkl`.
* Ưu tiên nạp tập dữ liệu đầy đủ đặc trưng (`X_train.parquet` hoặc `train.csv`).
* Cơ chế tự động ép kiểu dữ liệu (*Downcasting Numeric Types*) giúp tiết kiệm tới **50% RAM** khi xử lý lưu lượng mạng lớn.


In [ ]:
# ==============================================================================
# 2. DATA INGESTION & MEMORY MANAGEMENT
# ==============================================================================
DATA_CANDIDATE_PATHS = [
    '/kaggle/input/ciciot2023/',             # Kaggle dataset
    '/kaggle/input/cic-iot-2023/',          # Kaggle alt
    '/content/drive/MyDrive/do_an/data/',   # Google Colab Drive
    './data/',                              # Local project root
    '../data/',                             # Parent dir
    './'                                    # Current dir
]

def resolve_data_directory():
    for p in DATA_CANDIDATE_PATHS:
        if os.path.exists(p) and any(os.scandir(p)):
            return p
    return './data/'

DATA_ROOT = resolve_data_directory()
print(f"📁 Root data path resolved: {os.path.abspath(DATA_ROOT)}")

def optimize_memory_usage(df):
    """Downcast float64 and int64 columns to save 50%+ RAM."""
    start_mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif col_type == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    end_mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"💾 Memory reduced from {start_mem:.2f} MB to {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% saved)")
    return df

def load_dataset_sample(data_dir, max_rows=500000):
    """Load dataset from CSV, Parquet, or Pickle with adaptive fallback."""
    csv_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.csv')]
    parquet_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.parquet')]
    pkl_x_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.pkl') and ('X_train' in f or 'X_' in f)]

    def prioritize(files, keywords):
        for kw in keywords:
            matched = [f for f in files if kw in os.path.basename(f)]
            if matched:
                return matched[0]
        return files[0] if files else None

    df = None
    if csv_files:
        target_csv = prioritize(csv_files, ['train.csv', 'ciciot', 'dataset', '.csv'])
        print(f"📥 Loading CSV: {target_csv} (max {max_rows:,} rows)...")
        df = pd.read_csv(target_csv, nrows=max_rows, low_memory=False)
    elif parquet_files:
        # Prioritize full dataset (X_train.parquet) over selected (X_train_selected.parquet)
        target_pq = prioritize(parquet_files, ['X_train.parquet', 'train.parquet', '.parquet'])
        print(f"📥 Loading Parquet: {target_pq}...")
        df = pd.read_parquet(target_pq)
        if len(df) > max_rows:
            df = df.sample(n=max_rows, random_state=RANDOM_STATE)
    elif pkl_x_files:
        target_pkl = prioritize(pkl_x_files, ['X_train.pkl', 'train.pkl', '.pkl'])
        print(f"📥 Loading Pickle with joblib: {target_pkl}...")
        df = joblib.load(target_pkl)
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)
        if len(df) > max_rows:
            df = df.sample(n=max_rows, random_state=RANDOM_STATE)
    else:
        raise FileNotFoundError(f"No suitable CSV, Parquet, or PKL data found under {data_dir}!")

    # Attach labels if saved separately in y_train.pkl
    if 'label' not in df.columns and 'attack_group' not in df.columns:
        y_candidates = [
            os.path.join(data_dir, 'y_train.pkl'),
            os.path.join(data_dir, 'y_train_balanced.pkl')
        ]
        for y_f in y_candidates:
            if os.path.exists(y_f):
                print(f"🔗 Attaching target labels from: {y_f}")
                try:
                    y_obj = joblib.load(y_f)
                except Exception:
                    y_obj = pd.read_pickle(y_f)
                if isinstance(y_obj, dict):
                    y_vals = y_obj.get('label_name', y_obj.get('target_name', y_obj.get('category_name', list(y_obj.values())[0])))
                else:
                    y_vals = y_obj
                df['label'] = list(y_vals[:len(df)])
                break

    df = optimize_memory_usage(df)
    return df

# Load working dataframe
df = load_dataset_sample(DATA_ROOT, max_rows=500000)
print(f"\n✅ Successfully loaded working dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")


<a id="sec-03"></a>
## 3. Kiểm toán Chất lượng Dữ liệu (Data Quality Audit)
* Kiểm tra tính toàn vẹn của tập dữ liệu: giá trị khuyết thiếu (`NaN`, `null`), giá trị vô cực (`inf`, `-inf`), số lượng bản ghi trùng lặp (`duplicates`).
* Thống kê tỷ lệ các đặc trưng "Zero-Inflated" (chứa phần lớn giá trị 0) thường gặp ở các cờ TCP hiếm khi được bật.


In [ ]:
# ==============================================================================
# 3. DATA QUALITY AUDIT
# ==============================================================================
print("=" * 75)
print("🔍 DATA QUALITY AUDIT & INTEGRITY CHECK")
print("=" * 75)

# 1. Duplicate records
n_duplicates = df.duplicated().sum()
print(f"📋 Duplicate Rows: {n_duplicates:,} ({n_duplicates / len(df) * 100:.2f}%)")

# 2. Missing & Infinite Values
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
missing_counts = df[num_cols].isna().sum()
inf_counts = df[num_cols].apply(lambda c: np.isinf(c).sum())
zero_counts = (df[num_cols] == 0).sum()

quality_df = pd.DataFrame({
    'Feature': num_cols,
    'Missing_Count': missing_counts,
    'Missing_Pct': (missing_counts / len(df) * 100).round(2),
    'Inf_Count': inf_counts,
    'Inf_Pct': (inf_counts / len(df) * 100).round(2),
    'Zero_Pct': (zero_counts / len(df) * 100).round(2)
}).sort_values('Zero_Pct', ascending=False)

print("\nTop 15 Zero-Inflated Features:")
print(quality_df[['Feature', 'Missing_Pct', 'Inf_Pct', 'Zero_Pct']].head(15).to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Zero percentages
top_zeros = quality_df.head(15)
sns.barplot(data=top_zeros, x='Zero_Pct', y='Feature', ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 15 Zero-Inflated Features (%)', fontweight='bold')
axes[0].set_xlabel('Percentage of Zeros (%)')

# Missing / Inf bar
top_inf = quality_df.sort_values('Inf_Pct', ascending=False).head(10)
sns.barplot(data=top_inf, x='Inf_Pct', y='Feature', ax=axes[1], palette='Reds_r')
axes[1].set_title('Top Features with Infinite Values (%)', fontweight='bold')
axes[1].set_xlabel('Percentage of Inf (%)')

plt.tight_layout()
plt.show()

# Clean infinities for downstream analysis
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
print("\n✅ Cleaned inf/-inf and NaN with robust median replacement for analysis.")


<a id="sec-04"></a>
## 4. Phân tích Phân phối Nhãn Tấn công & Độ Mất Cân Bằng Nghiêm Trọng (34 → 8 Classes)
* Bài toán CICIoT2023 gốc có **34 nhãn tấn công chi tiết** thuộc các nhóm: DoS, DDoS, Reconnaissance, Web-based, Brute Force, Spoofing, Mirai, và Benign.
* Theo bài báo IEEE Access chuẩn (*"Toward Enhanced Attack Detection and Explanation in Intrusion Detection System-Based IoT Environment Data"*), 34 nhãn được ánh xạ thành **8 nhóm hành vi chính**.
* Đo lường **Hệ số Mất cân bằng (Imbalance Ratio)** và đánh giá đường cong Pareto.


In [ ]:
# ==============================================================================
# 4. ATTACK TAXONOMY (34 -> 8 CLASSES) & IMBALANCE DYNAMICS
# ==============================================================================
# IEEE Access 2023 Taxonomy Mapping
ATTACK_MAPPING_34_TO_8 = {
    # 1. DDoS attacks
    'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-HTTP_Flood': 'DDoS',
    'DDoS-ICMP_Flood': 'DDoS',
    'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-PSHACK_Flood': 'DDoS',
    'DDoS-RSTFINFlood': 'DDoS',
    'DDoS-SYN_Flood': 'DDoS',
    'DDoS-SlowLoris': 'DDoS',
    'DDoS-SynonymousIP_Flood': 'DDoS',
    'DDoS-TCP_Flood': 'DDoS',
    'DDoS-UDP_Flood': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS',
    'DDoS': 'DDoS',
    
    # 2. DoS attacks
    'DoS-HTTP_Flood': 'DoS',
    'DoS-SYN_Flood': 'DoS',
    'DoS-TCP_Flood': 'DoS',
    'DoS-UDP_Flood': 'DoS',
    'DoS': 'DoS',
    
    # 3. Mirai Botnet
    'Mirai-greeth_flood': 'Mirai',
    'Mirai-greip_flood': 'Mirai',
    'Mirai-udpplain': 'Mirai',
    'Mirai': 'Mirai',
    
    # 4. Reconnaissance
    'Recon-HostDiscovery': 'Recon',
    'Recon-OSScan': 'Recon',
    'Recon-PingSweep': 'Recon',
    'Recon-PortScan': 'Recon',
    'VulnerabilityScan': 'Recon',
    'Recon': 'Recon',
    
    # 5. Spoofing
    'DNS_Spoofing': 'Spoofing',
    'MITM-ArpSpoofing': 'Spoofing',
    'Spoofing': 'Spoofing',
    
    # 6. Web-based attacks
    'Backdoor_Malware': 'Web-Based',
    'BrowserHijacking': 'Web-Based',
    'CommandInjection': 'Web-Based',
    'SqlInjection': 'Web-Based',
    'Uploading_Attack': 'Web-Based',
    'XSS': 'Web-Based',
    'Web': 'Web-Based',
    'Web-Based': 'Web-Based',
    
    # 7. Brute Force
    'DictionaryBruteForce': 'BruteForce',
    'BruteForce': 'BruteForce',
    
    # 8. Benign / Normal
    'BenignTraffic': 'Normal',
    'Benign': 'Normal',
    'Normal': 'Normal'
}

# Identify raw label column
label_col_candidate = next((c for c in df.columns if c.lower() in ['label', 'target', 'attack_type', 'class']), None)
if not label_col_candidate:
    label_col_candidate = df.columns[-1]

df['attack_group'] = df[label_col_candidate].astype(str).map(lambda x: ATTACK_MAPPING_34_TO_8.get(x.strip(), 'Web-Based'))

# Compute Distributions
raw_counts = df[label_col_candidate].value_counts()
group_counts = df['attack_group'].value_counts()

max_class = group_counts.max()
min_class = group_counts.min()
imbalance_ratio = max_class / min_class

print(f"📌 Raw Label Column: '{label_col_candidate}' with {len(raw_counts)} unique labels")
print(f"⚖️  8-Class Imbalance Ratio (Max / Min): {imbalance_ratio:.2f} : 1")
print(f"   - Lớp đa số lớn nhất ({group_counts.index[0]}): {max_class:,} mẫu ({max_class/len(df)*100:.2f}%)")
print(f"   - Lớp thiểu số nhỏ nhất ({group_counts.index[-1]}): {min_class:,} mẫu ({min_class/len(df)*100:.2f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Subplot 1: 8 Classes Distribution
colors = [ATTACK_PALETTE.get(c, '#95a5a6') for c in group_counts.index]
bars = axes[0].bar(group_counts.index, group_counts.values, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Phân phối 8 Nhóm Lớp Tấn công (IEEE Access Taxonomy)', fontweight='bold')
axes[0].set_ylabel('Số lượng mẫu (Samples)')
axes[0].tick_params(axis='x', rotation=30)
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, yval + len(df)*0.01, f"{yval:,}\n({yval/len(df)*100:.1f}%)", 
                 ha='center', va='bottom', fontsize=8, fontweight='bold')

# Subplot 2: Donut Chart
axes[1].pie(group_counts.values, labels=group_counts.index, colors=colors, autopct='%1.1f%%',
            startangle=140, pctdistance=0.75, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].add_artist(plt.Circle((0,0), 0.50, fc='white'))
axes[1].set_title('Tỷ lệ Phần trăm Các Nhóm Lưu lượng (Donut View)', fontweight='bold')

plt.tight_layout()
plt.show()


<a id="sec-05"></a>
## 5. Phân tích Tốc độ Dòng & Hành vi Tấn công Lưu lượng Khối lượng lớn (Rate, Srate, Drate)
* Các cuộc tấn công từ chối dịch vụ (**DDoS/DoS**) có bản chất là cố gắng áp đảo tài nguyên băng thông và CPU của thiết bị IoT.
* Các chỉ số đo lường then chốt:
  - `Rate`: Tổng tốc độ gói tin trên giây (Packets per second).
  - `Srate`: Tốc độ gói gửi từ địa chỉ nguồn đến đích (Source to Destination).
  - `Drate`: Tốc độ gói phản hồi từ đích về nguồn (Destination to Source).
  - `flow_duration` / `Duration`: Thời gian tồn tại của phiên kết nối mạng.
* **Quy luật vật lý an ninh mạng:** Trong tấn công DDoS, lưu lượng một chiều áp đảo (Srate >> Drate), khiến thiết bị IoT bị tê liệt không thể gửi gói tin phản hồi.


In [ ]:
# ==============================================================================
# 5. FLOW RATES & VOLUMETRIC ATTACK SIGNATURES
# ==============================================================================
rate_features = [f for f in ['Rate', 'Srate', 'Drate', 'flow_duration', 'Duration'] if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]

print(f"📊 Analyzing Rate Features: {rate_features}")

# Compute Statistics across 8 classes
rate_summary = df.groupby('attack_group')[rate_features].agg(['mean', 'median', 'std'])
display_fn(rate_summary.round(2))

# Visualization with Symlog Scale
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feat in enumerate([f for f in ['Rate', 'Srate', 'Drate'] if f in df.columns][:3]):
    ax = axes[idx]
    sns.boxplot(data=df, x='attack_group', y=feat, ax=ax, palette=ATTACK_PALETTE, showfliers=False)
    ax.set_yscale('symlog')
    ax.set_title(f'Phân bố {feat} (Symlog Scale)', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('So sánh Chỉ số Tốc độ Dòng (Flow Rates) giữa Lưu lượng Bình thường và Tấn công', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Asymmetry Feature Discovery
if 'Srate' in df.columns and 'Drate' in df.columns:
    df['traffic_asymmetry'] = (df['Srate'] - df['Drate']) / (df['Srate'] + df['Drate'] + 1e-5)
    plt.figure(figsize=(10, 4))
    sns.kdeplot(data=df, x='traffic_asymmetry', hue='attack_group', common_norm=False, palette=ATTACK_PALETTE, alpha=0.5)
    plt.title('Độ Bất Đối Xứng Chiều Gửi/Nhận: (Srate - Drate) / (Srate + Drate)', fontweight='bold')
    plt.xlabel('Asymmetry Score (-1.0 = Phản hồi chiếm ưu thế, +1.0 = Bơm lưu lượng một chiều)')
    plt.show()


<a id="sec-06"></a>
## 6. Giải phẫu Chữ ký Cờ TCP & Giao thức Mạng (TCP Flags Heatmap & Profile)
* Giao thức TCP dựa trên cơ chế bắt tay 3 bước (*Three-way Handshake*): `SYN` → `SYN-ACK` → `ACK`.
* Kẻ tấn công lợi dụng các cờ TCP để thực hiện các cuộc tấn công tinh vi:
  - **SYN Flood**: Gửi ồ ạt cờ `SYN` nhưng không bao giờ hoàn tất bắt tay (bỏ qua bước `ACK`).
  - **PSH-ACK Flood**: Đẩy cờ `PSH` (Push) để ép thiết bị đích giải phóng bộ đệm ứng dụng ngay lập tức.
  - **RST/FIN Flood / PortScan**: Gửi cờ `RST` (Reset) hoặc `FIN` (Finish) để dò tìm các cổng mở mà không để lại log hoàn chỉnh.
* Phân tích Ma trận Nhiệt (**TCP Flag Heatmap**) theo từng loại tấn công.


In [ ]:
# ==============================================================================
# 6. TCP FLAGS & PROTOCOL BEHAVIORAL HEATMAP
# ==============================================================================
# Non-feature / metadata columns that must NEVER be treated as TCP flags
exclude_target_cols = {'attack_group', 'label', 'target', 'attack_type', 'is_attack', 'is_malicious', 'traffic_asymmetry'}

# Strict detection of TCP flag and count features (prevent 'ack' from matching 'attack_group')
flag_keywords = ['_flag_', '_count', 'flag_number', 'total_flags', 'total_counts', 'syn_ack_ratio', 'rst_syn_ratio']
flag_cols = [
    c for c in df.columns
    if c not in exclude_target_cols
    and pd.api.types.is_numeric_dtype(df[c])
    and (
        any(k in c.lower() for k in flag_keywords)
        or any(c.lower().startswith(f) for f in ['fin_', 'syn_', 'rst_', 'psh_', 'ack_', 'ece_', 'cwr_', 'urg_'])
    )
]
print(f"🚩 Discovered TCP Flag features: {flag_cols}")

if flag_cols:
    # Compute average flag intensity per attack group
    flag_profile = df.groupby('attack_group')[flag_cols].mean()

    plt.figure(figsize=(14, 6))
    sns.heatmap(flag_profile, cmap='YlOrRd', annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'label': 'Tần suất xuất hiện trung bình'})
    plt.title('Heatmap Chữ Ký Cờ TCP (TCP Flags Signature Profile) theo Từng Nhóm Tấn Công', fontweight='bold')
    plt.ylabel('Nhóm Tấn Công (Attack Group)')
    plt.xlabel('Đặc Trưng Cờ TCP (Flag Features)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Protocol Breakdown (L4 Protocols: TCP, UDP, ICMP)
proto_cols = [c for c in ['TCP', 'UDP', 'ICMP', 'HTTP', 'HTTPS', 'DNS', 'ARP', 'SMTP', 'SSH', 'DHCP'] if c in df.columns and pd.api.types.is_numeric_dtype(df[c]) and c not in exclude_target_cols]
if proto_cols:
    proto_summary = df.groupby('attack_group')[proto_cols].mean() * 100
    print()
    print("🌐 Tỷ Lệ Giao Thức L4/L7 (%) Trong Từng Nhóm Tấn Công:")
    display_fn(proto_summary.round(1))


<a id="sec-07"></a>
## 7. Phân phối Kích thước Gói tin & Độ Phân tán Thống kê (Packet Size & Statistical Dispersion)
* Lưu lượng mạng IoT thông thường gồm các gói cảm biến nhỏ (telemetry) đan xen với các gói payload lớn (firmware update, HTTP API). Do đó độ lệch chuẩn kích cỡ gói tin (`Std`, `Variance`) tương đối phân tán.
* Ngược lại, các công cụ DoS/DDoS tự động sinh ra các gói tin kích thước cố định để tối ưu hóa tốc độ tạo gói (`Std` ≈ 0).
* Khảo sát các đại lượng: `Tot sum`, `Min`, `Max`, `AVG`, `Std`, `Tot size`, `Magnitue`, `Radius`, `Variance`.


In [ ]:
# ==============================================================================
# 7. PACKET SIZE DISTRIBUTION & STATISTICAL DISPERSION
# ==============================================================================
size_features = [f for f in ['Tot sum', 'Tot size', 'Min', 'Max', 'AVG', 'Std', 'Variance', 'Magnitue', 'Radius'] if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]
print(f"📏 Analyzing Packet Size Features: {size_features}")

if size_features:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    inspect_cols = [c for c in ['AVG', 'Std', 'Tot size', 'Variance'] if c in df.columns][:4]
    for idx, col in enumerate(inspect_cols):
        ax = axes[idx]
        sns.violinplot(data=df, x='attack_group', y=col, ax=ax, palette=ATTACK_PALETTE, cut=0)
        ax.set_yscale('symlog')
        ax.set_title(f'Phân Bố Độ Biến Thiên Kích Thước: {col} (Symlog)', fontweight='bold')
        ax.tick_params(axis='x', rotation=35)
        ax.set_xlabel('')
        
    plt.suptitle('Đặc Tính Thống Kê Kích Thước Gói Tin Giữa Các Nhóm Tấn Công', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


<a id="sec-08"></a>
## 8. Phân tích Thời gian Đến giữa các Gói tin (Inter-Arrival Time - IAT & Botnet Jitter)
* `IAT` (Inter-Arrival Time) đo lường khoảng thời gian trôi qua giữa hai gói tin liên tiếp trong một luồng mạng.
* **Ý nghĩa an ninh mạng:**
  - **Người dùng thật / Thiết bị IoT bình thường:** Có thời gian nghỉ giữa các phiên truyền dữ liệu, đồ thị IAT trải rộng theo phân phối Poisson/Pareto.
  - **Mã độc Botnet (như Mirai) hoặc Volumetric Flooding:** Máy tấn công chạy vòng lặp gửi gói liên tục không trễ, dẫn đến IAT hội tụ về các giá trị cực kỳ nhỏ và đồng đều.


In [ ]:
# ==============================================================================
# 8. INTER-ARRIVAL TIME (IAT) ANALYSIS
# ==============================================================================
iat_cols = [c for c in df.columns if 'iat' in c.lower() and c not in exclude_target_cols and pd.api.types.is_numeric_dtype(df[c])]
print(f"⏱️ Discovered IAT Features: {iat_cols}")

if iat_cols:
    target_iat = iat_cols[0]
    
    plt.figure(figsize=(14, 5))
    for group in df['attack_group'].unique():
        raw_vals = pd.to_numeric(df[df['attack_group'] == group][target_iat], errors='coerce').dropna()
        subset = raw_vals[raw_vals > 0]
        if len(subset) > 5:
            sns.kdeplot(subset, label=group, log_scale=True, common_norm=False, color=ATTACK_PALETTE.get(group, None), alpha=0.6, linewidth=2)
            
    plt.title(f'Hàm Mật Độ Xác Suất Thời Gian Đến Giữa Các Gói Tin ({target_iat}) theo Log Scale', fontweight='bold')
    plt.xlabel(f'{target_iat} (Microseconds / Log Scale)')
    plt.ylabel('Density (Mật độ)')
    plt.legend(title='Attack Group')
    plt.tight_layout()
    plt.show()
    
    print("\nThống kê IAT theo từng nhóm tấn công:")
    display_fn(df.groupby('attack_group')[target_iat].describe()[['mean', 'std', 'min', '50%', 'max']].round(2))


<a id="sec-09"></a>
## 9. Đánh giá Sức mạnh Phân tách (Effect Size / Cohen's d: Benign vs Attacks)
* **Cohen's $d$** là thước đo kích thước hiệu ứng (*Effect Size*) chuẩn hóa sự khác biệt giữa hai phân phối:
  $$d = \frac{|\mu_{\text{Attack}} - \mu_{\text{Normal}}|}{\sqrt{(\sigma_{\text{Attack}}^2 + \sigma_{\text{Normal}}^2) / 2}}$$
* **Ngưỡng đánh giá học máy:**
  - $d > 0.8$: Sức mạnh phân tách lớn (*Large Effect*) - Đặc trưng cốt lõi cho mô hình.
  - $0.5 < d \le 0.8$: Sức mạnh trung bình (*Medium Effect*).
  - $d < 0.2$: Khả năng phân biệt yếu (*Negligible*).


In [ ]:
# ==============================================================================
# 9. DISCRIMINATIVE POWER ANALYSIS (COHEN'S D EFFECT SIZE)
# ==============================================================================
# Binary label: Normal vs Attack
df['is_malicious'] = (df['attack_group'] != 'Normal').astype(int)

benign_sample = df[df['is_malicious'] == 0]
attack_sample = df[df['is_malicious'] == 1]

exclude_analysis_cols = {'is_malicious', 'traffic_asymmetry', 'attack_group', 'label', 'target', 'attack_type', 'is_attack', 'label_encoded', 'category_encoded'}
feature_candidates = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_analysis_cols]

cohen_records = []
for col in feature_candidates:
    b_vals = benign_sample[col].dropna()
    a_vals = attack_sample[col].dropna()
    
    if len(b_vals) < 10 or len(a_vals) < 10:
        continue
    
    m_b, m_a = b_vals.mean(), a_vals.mean()
    var_b, var_a = b_vals.var(), a_vals.var()
    pooled_sd = np.sqrt((var_b + var_a) / 2.0)
    
    if pooled_sd > 1e-9:
        d_val = abs(m_a - m_b) / pooled_sd
    else:
        d_val = 0.0
        
    cohen_records.append({
        'Feature': col,
        'Benign_Mean': m_b,
        'Attack_Mean': m_a,
        'Cohen_d': d_val,
        'Effect_Size': 'Large (d > 0.8)' if d_val > 0.8 else ('Medium (0.5-0.8)' if d_val > 0.5 else 'Small (< 0.5)')
    })

effect_df = pd.DataFrame(cohen_records).sort_values('Cohen_d', ascending=False)

print(f"Top 15 Đặc Trưng Có Khả Năng Phân Biệt Tấn Công Mạnh Nhất (Cohen's d):")
display_fn(effect_df[['Feature', 'Cohen_d', 'Effect_Size', 'Benign_Mean', 'Attack_Mean']].head(15).round(3))

# Diverging Horizontal Bar Chart
top_effect = effect_df.head(20)
plt.figure(figsize=(12, 6))
bar_colors = ['#27ae60' if e.startswith('Large') else ('#f39c12' if e.startswith('Medium') else '#e74c3c') for e in top_effect['Effect_Size']]
plt.barh(top_effect['Feature'][::-1], top_effect['Cohen_d'][::-1], color=bar_colors[::-1], edgecolor='black', alpha=0.85)
plt.axvline(0.8, color='red', linestyle='--', linewidth=1.5, label="Ngưỡng phân tách mạnh (d=0.8)")
plt.axvline(0.5, color='orange', linestyle=':', linewidth=1.5, label="Ngưỡng trung bình (d=0.5)")
plt.xlabel("Cohen's d Effect Size")
plt.title("Xếp Hạng Các Đặc Trưng Phân Biệt Giữa Lưu Lượng Lành Tính và Độc Hại", fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


<a id="sec-10"></a>
## 10. Phân tích Đa cộng tuyến, Ma trận Tương quan & Phân cụm Phân cấp (Clustermap & Dendrogram)
* Các gói tin mạng sinh ra nhiều biến số phụ thuộc tuyến tính (ví dụ: `Tot sum` tỷ lệ thuận với `Tot size`, `Rate` tỷ lệ với `Srate`).
* Đa cộng tuyến (*Multicollinearity*) gây nhiễu cho mô hình học máy và làm phân tán độ quan trọng trong diễn giải giải thích mô hình (XAI).
* Biểu đồ **Clustermap** kết hợp cây phân cấp (*Hierarchical Clustering Dendrogram*) giúp phát hiện các cụm đặc trưng đồng dạng.


In [ ]:
# ==============================================================================
# 10. CORRELATION CLUSTERMAP & DENDROGRAM
# ==============================================================================
# Select numeric features
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_analysis_cols]
sample_corr = df[numeric_cols].sample(n=min(50000, len(df)), random_state=RANDOM_STATE)

corr_matrix = sample_corr.corr().abs()

# Find highly collinear pairs (|r| > 0.90)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        val = corr_matrix.iloc[i, j]
        if val >= 0.90:
            high_corr_pairs.append({'Feature_1': col1, 'Feature_2': col2, 'Correlation': round(val, 4)})

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
print(f"⚠️ Phát hiện {len(high_corr_df)} cặp đặc trưng có tương quan cực cao (|r| >= 0.90):")
display_fn(high_corr_df.head(15))

# Hierarchical Clustering Clustermap
corr_filled = corr_matrix.fillna(0)
cluster_grid = sns.clustermap(corr_filled, cmap='mako', figsize=(14, 12), dendrogram_ratio=(0.15, 0.15),
                              cbar_kws={'label': 'Pearson Correlation |r|'}, linewidths=0.2)
cluster_grid.fig.suptitle('Clustermap Ma Trận Tương Quan & Cây Phân Cụm Phân Cấp (Dendrogram)', y=1.02, fontweight='bold', fontsize=14)
plt.show()


<a id="sec-11"></a>
## 11. Xếp hạng Tầm quan trọng Đặc trưng Đồng thuận (Consensus Feature Importance)
* Để lựa chọn 25 đặc trưng tinh hoa nhất như bài báo IEEE Access đề xuất, ta sử dụng phương pháp **Consensus Importance**:
  1. **Random Forest MDI** (Mean Decrease in Impurity / Gini Importance).
  2. **Extra Trees Classifier** (Tạo lát cắt ngẫu nhiên, kháng nhiễu và bất biến với đa cộng tuyến).
  3. **ANOVA F-value** (Thước đo phân tách thống kê đa lớp theo phương sai).
* Lấy trung bình chuẩn hóa để thu được bảng xếp hạng đồng thuận cao nhất.


In [ ]:
# ==============================================================================
# 11. CONSENSUS FEATURE IMPORTANCE (RF + EXTRA TREES + ANOVA F)
# ==============================================================================
sub_sample = df.sample(n=min(30000, len(df)), random_state=RANDOM_STATE)
X_sub = sub_sample[numeric_cols].fillna(0)
y_sub = LabelEncoder().fit_transform(sub_sample['attack_group'])

# 1. Random Forest Importance
print("🌲 Training Random Forest Selector...")
rf_sel = RandomForestClassifier(n_estimators=50, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
rf_sel.fit(X_sub, y_sub)
rf_imp = rf_sel.feature_importances_

# 2. Extra Trees Importance
print("🌳 Training Extra Trees Selector...")
et_sel = ExtraTreesClassifier(n_estimators=50, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
et_sel.fit(X_sub, y_sub)
et_imp = et_sel.feature_importances_

# 3. ANOVA F-Score
print("📐 Computing ANOVA F-scores...")
f_scores, _ = f_classif(X_sub, y_sub)
f_scores = np.nan_to_num(f_scores, 0)

# Normalize scores to [0, 1]
norm_rf = rf_imp / rf_imp.max()
norm_et = et_imp / et_imp.max()
norm_f = f_scores / f_scores.max()

consensus_scores = (norm_rf * 0.4 + norm_et * 0.4 + norm_f * 0.2)
importance_df = pd.DataFrame({
    'Feature': numeric_cols,
    'RF_Importance': rf_imp.round(4),
    'ET_Importance': et_imp.round(4),
    'ANOVA_F': f_scores.round(2),
    'Consensus_Score': consensus_scores.round(4)
}).sort_values('Consensus_Score', ascending=False)

print("\n🏆 Top 25 Đặc Trưng Quan Trọng Nhất (Consensus Selection):")
display_fn(importance_df.head(25))

# Plot Top-25
plt.figure(figsize=(12, 7))
top25 = importance_df.head(25)
sns.barplot(data=top25, x='Consensus_Score', y='Feature', palette='viridis')
plt.title('Top 25 Đặc Trưng Quan Trọng Nhất Đạt Đồng Thuận (Consensus Feature Importance)', fontweight='bold')
plt.xlabel('Điểm Đồng Thuận Chuẩn Hóa (Weighted Consensus Score)')
plt.tight_layout()
plt.show()


<a id="sec-12"></a>
## 12. Trực quan hóa Không gian Chiều cao (PCA Scree & t-SNE 2D Manifold Projection)
* Chiếu không gian 46+ chiều xuống không gian 2D để quan sát mức độ phân tách cụm giữa 8 nhóm tấn công.
* **PCA Scree Plot**: Đánh giá tỷ lệ phương sai giải thích tích lũy (*Cumulative Explained Variance*).
* **t-SNE 2D Projection**: Thuật toán phi tuyến tính bảo toàn cấu trúc lân cận cục bộ, làm nổi bật cụm Mirai, DDoS và Normal.


In [ ]:
# ==============================================================================
# 12. DIMENSIONALITY REDUCTION (PCA & T-SNE 2D PROJECTION)
# ==============================================================================
top_25_features = importance_df['Feature'].head(25).tolist()

manifold_sample = df.sample(n=min(3000, len(df)), random_state=RANDOM_STATE)
X_mani = StandardScaler().fit_transform(manifold_sample[top_25_features])
y_mani = manifold_sample['attack_group'].values

# 1. PCA Analysis
pca = PCA(n_components=10, random_state=RANDOM_STATE)
X_pca_10 = pca.fit_transform(X_mani)

# 2. t-SNE Analysis (Sampled 3,000 for fast interactive execution)
print("🎨 Computing t-SNE 2D embedding (3,000 samples)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X_mani)

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PCA Scree plot
cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
axes[0].bar(range(1, 11), pca.explained_variance_ratio_ * 100, color='#34495e', alpha=0.7, label='Phương sai từng thành phần (Individual)')
axes[0].plot(range(1, 11), cum_var, color='#e74c3c', marker='o', linewidth=2, label='Phương sai tích lũy (Cumulative)')
axes[0].axhline(80, color='gray', linestyle='--', label='Ngưỡng 80% thông tin')
axes[0].set_title('PCA Scree Plot: Tỷ Lệ Phương Sai Giải Thích Tích Lũy', fontweight='bold')
axes[0].set_xlabel('Số Thành Phần Chính (Principal Components)')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].legend(loc='best')

# Plot 2: t-SNE 2D Scatter
for group in np.unique(y_mani):
    idx = (y_mani == group)
    axes[1].scatter(X_tsne[idx, 0], X_tsne[idx, 1], c=ATTACK_PALETTE.get(group, 'gray'), 
                    label=group, alpha=0.6, s=15, edgecolor='none')
axes[1].set_title('t-SNE 2D Manifold: Không Gian Phân Tách Cụm 8 Nhóm Tấn Công', fontweight='bold')
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
axes[1].legend(title='Attack Group', markerscale=2, loc='best')

plt.tight_layout()
plt.show()


<a id="sec-13"></a>
## 13. Phân tích Dị biệt & Giá trị Ngoại lai (Outlier Behavior in Cybersecurity)
* Trong xử lý dữ liệu thông thường, Outlier thường bị coi là lỗi thu thập và bị loại bỏ.
* **Nguyên lý An ninh mạng đối kháng:**
  - Outlier trong an toàn thông tin **chính là dấu vết của các cuộc tấn công cực đoan** (Massive DDoS Floods, Zero-day probes, Buffer Overflow).
  - Loại bỏ mù quáng Outlier đồng nghĩa với việc xóa bỏ chính mẫu tấn công!
  - Thay vào đó, áp dụng các kỹ thuật co giãn bền vững (*Robust Scaling*) hoặc biến đổi logarit (*Log Transform*).


In [ ]:
# ==============================================================================
# 13. OUTLIER AUDIT & EXTREME VALUES BEHAVIOR
# ==============================================================================
outlier_metrics = []
for col in top_25_features[:15]:
    c_data = df[col].dropna()
    q1 = c_data.quantile(0.25)
    q3 = c_data.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    n_out = ((c_data < lower) | (c_data > upper)).sum()
    pct_out = (n_out / len(c_data)) * 100
    
    # Skewness & Kurtosis
    sk = skew(c_data)
    kt = kurtosis(c_data)
    
    outlier_metrics.append({
        'Feature': col,
        'IQR': round(iqr, 3),
        'Outlier_Count': n_out,
        'Outlier_Pct': round(pct_out, 2),
        'Skewness': round(sk, 2),
        'Kurtosis': round(kt, 2),
        'Treatment': 'Log Transform' if sk > 3 else ('Robust Scale' if pct_out > 10 else 'Standard Scale')
    })

outlier_table = pd.DataFrame(outlier_metrics).sort_values('Outlier_Pct', ascending=False)
print("📋 Bảng Phân Tích Độ Lệch (Skewness), Độ Nhọn (Kurtosis) & Tỷ Lệ Ngoại Lai:")
display_fn(outlier_table)

# Visualizing Outliers on Skewed Features
plt.figure(figsize=(14, 5))
sns.barplot(data=outlier_table, x='Feature', y='Outlier_Pct', palette='rocket')
plt.title('Tỷ Lệ Mẫu Ngoại Lai (%) Trên Các Đặc Trưng Quan Trọng Nhất', fontweight='bold')
plt.ylabel('Tỷ lệ ngoại lai (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


<a id="sec-14"></a>
## 14. Khai phá Kỹ thuật Trích xuất Đặc trưng Mới (Domain-Specific Feature Engineering)
* Dựa trên các phát hiện thống kê từ EDA, chúng tôi đề xuất 5 nhóm đặc trưng phái sinh mới mang tri thức an ninh mạng:
  1. **`syn_intensity`**: Tỷ lệ cờ `syn_count` hoặc `syn_flag_number` trên tổng số gói → Đột biến ở tấn công SYN Flood.
  2. **`rst_syn_ratio`**: Tỷ số `(rst_count + 1) / (syn_count + 1)` → Dấu hiệu nhận diện quét cổng (PortScan) và từ chối dịch vụ.
  3. **`payload_dispersion`**: Hệ số biến thiên kích thước gói `Std / (AVG + 1e-5)` → Phân biệt lưu lượng tự động với lưu lượng người dùng.
  4. **`traffic_asymmetry`**: Chỉ số bất đối xứng dòng `(Srate - Drate) / (Srate + Drate + 1e-5)` → Nhận diện dòng tấn công một chiều.
  5. **`flow_intensity`**: Cường độ luồng `Rate * Header_Length` → Cường độ tải áp lực lên router IoT.
* Đánh giá **sức mạnh phân tách gia tăng** của các đặc trưng mới được tạo ra bằng Cohen's $d$.


In [ ]:
# ==============================================================================
# 14. ADVANCED DOMAIN-SPECIFIC FEATURE ENGINEERING
# ==============================================================================
print("=" * 75)
print("🛠️ SYNTHESIZING NOVEL DOMAIN-INFORMED FEATURES")
print("=" * 75)

fe_df = df.copy()

# 1. SYN Ratio
if 'syn_count' in fe_df.columns and 'Number' in fe_df.columns:
    fe_df['FE_syn_ratio'] = fe_df['syn_count'] / (fe_df['Number'] + 1e-5)
elif 'syn_count' in fe_df.columns:
    fe_df['FE_syn_ratio'] = fe_df['syn_count'] / 100.0

# 2. RST to SYN Ratio (Port scanning signature)
if 'rst_count' in fe_df.columns and 'syn_count' in fe_df.columns:
    fe_df['FE_rst_syn_ratio'] = (fe_df['rst_count'] + 1.0) / (fe_df['syn_count'] + 1.0)

# 3. Payload Dispersion (Coefficient of Variation)
if 'Std' in fe_df.columns and 'AVG' in fe_df.columns:
    fe_df['FE_size_dispersion'] = fe_df['Std'] / (fe_df['AVG'] + 1e-5)

# 4. Traffic Asymmetry
if 'Srate' in fe_df.columns and 'Drate' in fe_df.columns:
    fe_df['FE_asymmetry'] = (fe_df['Srate'] - fe_df['Drate']) / (fe_df['Srate'] + fe_df['Drate'] + 1e-5)

# 5. Header Intensity
if 'Rate' in fe_df.columns and 'Header_Length' in fe_df.columns:
    fe_df['FE_header_intensity'] = fe_df['Rate'] * fe_df['Header_Length']

engineered_cols = [c for c in fe_df.columns if c.startswith('FE_')]
print(f"✨ Successfully created {len(engineered_cols)} novel engineered features: {engineered_cols}")

# Measure Separation Power of Engineered Features
fe_cohen = []
b_sub = fe_df[fe_df['is_malicious'] == 0]
a_sub = fe_df[fe_df['is_malicious'] == 1]

for ec in engineered_cols:
    b_val = b_sub[ec].dropna()
    a_val = a_sub[ec].dropna()
    sd = np.sqrt((b_val.var() + a_val.var()) / 2.0)
    d = abs(a_val.mean() - b_val.mean()) / (sd + 1e-9)
    fe_cohen.append({'Engineered_Feature': ec, 'Cohen_d_Separation': round(d, 4)})

fe_eval_df = pd.DataFrame(fe_cohen).sort_values('Cohen_d_Separation', ascending=False)
display_fn(fe_eval_df)

# Visualizing Engineered Features
if engineered_cols:
    fig, axes = plt.subplots(1, len(engineered_cols), figsize=(5 * len(engineered_cols), 4))
    if len(engineered_cols) == 1:
        axes = [axes]
    for idx, ec in enumerate(engineered_cols):
        sns.boxplot(data=fe_df, x='attack_group', y=ec, ax=axes[idx], palette=ATTACK_PALETTE, showfliers=False)
        axes[idx].set_title(f'{ec}', fontweight='bold')
        axes[idx].tick_params(axis='x', rotation=45)
    plt.suptitle('Đặc Tính Phân Tách Của Các Đặc Trưng Kỹ Thuật Mới Tạo (Engineered Features)', y=1.05, fontweight='bold')
    plt.tight_layout()
    plt.show()


<a id="sec-15"></a>
## 15. Tổng kết Tri thức Miền & Bảng Chữ ký Đặc trưng Tấn công
* **Bảng Ma trận Nhận diện Chữ ký Tấn công (Attack Signature Behavior Matrix)** tổng hợp toàn diện các phát hiện EDA cho 8 nhóm lưu lượng:

| Nhóm Tấn Công | Tốc Độ Dòng (`Rate`/`Srate`) | Hành Vi Cờ TCP | Phân Tán Kích Thước (`Std`) | Thời Gian Đến (`IAT`) | Tri Thức Miền An Ninh Mạng |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **DDoS** | **Cực cao** (> 10^4 pps) | Thiên lệch mạnh (`SYN`, `PSH-ACK`, `UDP`) | Cố định, `Std` cực nhỏ | IAT → 0 (liên tục) | Lưu lượng khối lượng lớn áp đảo băng thông |
| **DoS** | **Cao** | Tương tự DDoS nhưng từ 1 nguồn duy nhất | Thấp | Cực nhỏ | Tấn công cạn kiệt tài nguyên từ máy trạm đơn |
| **Recon** | Trung bình / Thấp | Đột biến cờ `RST`, `FIN`, `SYN` rải rác | Rất nhỏ (gói thăm dò) | Khoảng cách đều đặn | Quét cổng, thăm dò lỗ hổng dịch vụ IoT |
| **Mirai** | Cao / Đột biến | Gói tin `UDP Plain`, `GRE-IP` áp đảo | Rất thấp | Dày đặc, theo chu kỳ botnet | Mã độc IoT chiếm quyền điều khiển thiết bị |
| **Spoofing**| Biến thiên theo phiên | Cờ `DNS`/`ARP` phản hồi giả mạo | Trung bình | Biến động | Giả mạo danh tính để tấn công Man-in-the-Middle |
| **Web-Based**| Thấp / Bình thường | Chủ yếu cờ `HTTP`, `PSH-ACK` ứng dụng | Đa dạng theo payload web | Theo tương tác ứng dụng | Chèn mã độc SQLi, XSS, Uploading webshell |
| **BruteForce**| Tần suất đều, lặp lại | Chuỗi bắt tay TCP mở rồi đóng liên tục | Gói nhỏ chứa từ điển mật khẩu | Cố định theo script bẻ khóa | Dò quét mật khẩu xác thực SSH/Telnet/Web |
| **Normal** | Dao động tự nhiên | Cân bằng theo bắt tay 3 bước chuẩn | Phân tán mạnh | Phân phối ngẫu nhiên | Lưu lượng hợp lệ từ người dùng và cảm biến IoT |

---
### 💡 Khuyến nghị cho Giai đoạn Tiền xử lý & Huấn luyện Mô hình:
1. **Lựa chọn đặc trưng:** Tập trung vào **Top 25 đặc trưng Consensus** kết hợp các đặc trưng bất đối xứng (`traffic_asymmetry`) và tỷ lệ cờ (`rst_syn_ratio`).
2. **Xử lý mất cân bằng:** Lớp DDoS và DoS chiếm đa số áp đảo; bắt buộc phải kết hợp **Random Under-Sampling (RUS)** cho lớp đa số và **SMOTE** cho lớp thiểu số (Normal, BruteForce).
3. **Thang đo chuẩn hóa:** Sử dụng `RobustScaler` hoặc biến đổi `symlog` cho các đặc trưng tốc độ và kích thước gói để bảo toàn các mẫu ngoại lai đại diện cho tấn công.


In [ ]:
# ==============================================================================
# 15. SAVE EDA SUMMARY ARTIFACTS
# ==============================================================================
print("=" * 75)
print("🎉 TOÀN BỘ QUY TRÌNH EDA & TRỰC QUAN HÓA CHUYÊN SÂU ĐÃ HOÀN THÀNH!")
print("=" * 75)

# Save summary tables to local disk
os.makedirs('eda_results', exist_ok=True)
importance_df.head(25).to_csv('eda_results/top25_consensus_features.csv', index=False)
effect_df.head(25).to_csv('eda_results/top25_cohen_discriminative_features.csv', index=False)
high_corr_df.to_csv('eda_results/high_collinear_pairs.csv', index=False)

print("\n📁 Đã lưu trữ các bảng phân tích thống kê vào thư mục: ./eda_results/")
print("  1. top25_consensus_features.csv")
print("  2. top25_cohen_discriminative_features.csv")
print("  3. high_collinear_pairs.csv")
print("\n🚀 Sẵn sàng cung cấp tri thức miền an ninh mạng cho Luận văn và Báo cáo khoa học!")
